# Chapter 2: Classification — Predicting Categorical Outcomes

### 2.1 Logistic Regression for Churn Prediction

In [ ]:
np.random.seed(42)
n = 200
churn_df = pd.DataFrame({
    "Tenure_months": np.random.randint(1, 60, n),
    "Monthly_Charges": np.random.uniform(30, 120, n),
    "Support_Tickets": np.random.randint(0, 8, n)
})
logit = -3 + 0.04*churn_df["Monthly_Charges"] + 0.35*churn_df["Support_Tickets"] - 0.06*churn_df["Tenure_months"]
prob = 1 / (1 + np.exp(-logit))
churn_df["Churn"] = (np.random.rand(n) < prob).astype(int)

X = churn_df[["Tenure_months", "Monthly_Charges", "Support_Tickets"]]
y = churn_df["Churn"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

y_pred = log_reg.predict(X_test)
y_prob = log_reg.predict_proba(X_test)[:, 1]     # probability of class 1 (churn), not just the 0/1 label

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 3))

**`.predict()` vs. `.predict_proba()`** is a distinction worth internalizing early: `.predict()` gives you the final 0/1 (or class label) decision, using an implicit 50% probability threshold; `.predict_proba()` gives you the underlying probability itself. For most real business decisions, the probability is more useful than the raw label — you might want to flag "high risk" customers at a 30% churn probability threshold rather than 50%, depending on the cost of missing a churner vs. the cost of a false alarm, and that threshold choice should be a business decision, not scikit-learn's silent default.

### 2.2 Reading a Confusion Matrix and Classification Report

A **confusion matrix** for a binary problem is a 2×2 grid:

|  | Predicted: Stay | Predicted: Churn |
|---|---|---|
| **Actual: Stay** | True Negative (TN) | False Positive (FP) |
| **Actual: Churn** | False Negative (FN) | True Positive (TP) |

From these four numbers, the classification report derives the metrics that actually matter for a business decision:

- **Precision** = TP / (TP + FP) — "of everyone I flagged as a churn risk, how many actually churned?" High precision matters when acting on a false positive is *expensive* (e.g., an expensive retention offer sent to someone who was never going to leave).
- **Recall** = TP / (TP + FN) — "of everyone who actually churned, how many did I catch?" High recall matters when *missing* a true case is expensive (e.g., failing to intervene on a customer who was about to leave).
- **F1-score** — the harmonic mean of precision and recall, a single number balancing both when neither error type is obviously more important.

⚠️ **Accuracy alone is a dangerous metric for imbalanced problems**, which describes most real churn/fraud/default datasets — if only 5% of customers churn, a model that *always* predicts "will not churn" achieves 95% accuracy while being completely useless. Always check precision, recall, and the confusion matrix directly, and always ask which type of error (false positive or false negative) is more costly to *your specific business*, then optimize for the metric that reflects that.

**ROC-AUC** (Area Under the Receiver Operating Characteristic Curve) summarizes a model's ability to *rank* positive cases above negative ones across every possible probability threshold, in one number from 0.5 (no better than random) to 1.0 (perfect separation) — genuinely useful specifically because it's threshold-independent, letting you compare two models' underlying discriminative power before deciding on a specific business threshold.

### 2.3 Decision Trees — The Model You Can Explain in a Meeting

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)
print("Decision Tree Accuracy:", accuracy_score(y_test, tree.predict(X_test)))

importance = pd.Series(tree.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nFeature Importance:\n", importance)

A decision tree makes predictions through a sequence of simple yes/no splits ("Is Monthly_Charges > ₹75? If yes, is Support_Tickets > 3?..."), which you can literally draw as a flowchart and walk a non-technical stakeholder through directly — a genuinely rare and valuable property compared to most other model types. `max_depth=3` deliberately limits how many splits deep the tree can go; a shallower tree is more interpretable and less prone to overfitting, at some cost to raw predictive accuracy — exactly the kind of interpretability-vs-accuracy tradeoff you'll navigate constantly as a business-facing analyst, and often a case where the more explainable model is the *right* choice even at a small accuracy cost, specifically because stakeholders need to trust and act on it.

`feature_importances_` ranks which inputs the tree relied on most — this is frequently the single most business-valuable output of the entire modeling exercise, because "what drives churn" is often a more actionable answer than "here's a churn probability for each customer."

### 2.4 K-Nearest Neighbors (KNN)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
print("KNN Accuracy:", accuracy_score(y_test, knn.predict(X_test)))

KNN makes a prediction for a new data point by finding the `k` most similar existing data points (its "nearest neighbors," by straight-line distance across the features) and taking a majority vote of their labels. It's conceptually the simplest model in this entire volume — there's no real "training" beyond storing the data — but it's sensitive to feature scale (a feature ranging 0–100,000 will dominate the distance calculation over one ranging 0–1 unless you scale first, per Chapter 4) and can be slow to predict on very large datasets, since every prediction requires comparing against the entire stored training set.

### Cheat Sheet — Classification

| Task | Code |
|---|---|
| Logistic regression | `LogisticRegression(max_iter=1000).fit(X_train, y_train)` |
| Get probabilities | `model.predict_proba(X_test)[:, 1]` |
| Decision tree | `DecisionTreeClassifier(max_depth=3).fit(...)` |
| Feature importance | `model.feature_importances_` |
| KNN | `KNeighborsClassifier(n_neighbors=5).fit(...)` |
| Confusion matrix | `confusion_matrix(y_test, y_pred)` |
| Full report | `classification_report(y_test, y_pred)` |
| ROC-AUC | `roc_auc_score(y_test, y_prob)` |

---